# 02 — Customer Aggregation

Giữ **hai bảng**: (1) tổng hợp toàn lịch sử cho EDA; (2) bảng khách hàng dùng cho mô hình chỉ lấy dữ liệu trong `[t-L, t)`. Phương án FINAL đang thử: `t = 01/06/2011`, `L = 365 ngày`, khách đủ điều kiện phải có ít nhất một lần mua trong 180 ngày trước `t`.

In [1]:
import pandas as pd
import numpy as np
import os

input_path = '../data/interim/transactions_cleaned.csv'
output_eda = '../data/interim/customer_base.csv'
output_model = '../data/interim/customer_base_model.csv'

df = pd.read_csv(input_path, parse_dates=['InvoiceDate'])
print('Số dòng cleaned:', len(df))
print('Khoảng ngày:', df['InvoiceDate'].min(), '->', df['InvoiceDate'].max())

Số dòng cleaned: 794454
Khoảng ngày: 2009-12-01 07:45:00 -> 2011-12-09 12:50:00


In [2]:
# 1) BẢNG TOÀN LỊCH SỬ — chỉ phục vụ EDA/mô tả cuối kỳ
purchases_all = df[df['transaction_type'] == 'purchase'].copy()

base_all = purchases_all.groupby('Customer ID').agg(
    Total_Invoices=('Invoice', 'nunique'),
    First_Purchase_Date=('InvoiceDate', 'min'),
    Last_Purchase_Date=('InvoiceDate', 'max')
).reset_index()

net_all = df.groupby('Customer ID').agg(
    Total_Spend=('Revenue', 'sum'),
    Total_Items=('Quantity', 'sum')
).reset_index()

customer_all = base_all.merge(net_all, on='Customer ID', how='left')
customer_all.to_csv(output_eda, index=False)
print('Đã lưu bảng EDA:', output_eda, '| khách:', len(customer_all))

Đã lưu bảng EDA: ../data/interim/customer_base.csv | khách: 5853


In [3]:
# 2) BẢNG DÙNG CHO MÔ HÌNH — chỉ nhìn quá khứ trước t
prediction_date = pd.Timestamp('2011-06-01')
lookback_days = 365
history_start = prediction_date - pd.Timedelta(days=lookback_days)
active_window_days = 180
active_start = prediction_date - pd.Timedelta(days=active_window_days)

hist = df[(df['InvoiceDate'] >= history_start) & (df['InvoiceDate'] < prediction_date)].copy()
purchases_hist = hist[hist['transaction_type'] == 'purchase'].copy()

# Khách đủ điều kiện: có mua hợp lệ trong 180 ngày trước t
eligible_ids = purchases_hist.loc[
    purchases_hist['InvoiceDate'] >= active_start, 'Customer ID'
].dropna().unique()

hist = hist[hist['Customer ID'].isin(eligible_ids)].copy()
purchases_hist = purchases_hist[purchases_hist['Customer ID'].isin(eligible_ids)].copy()

base_model = purchases_hist.groupby('Customer ID').agg(
    Frequency=('Invoice', 'nunique'),
    First_Purchase_Date=('InvoiceDate', 'min'),
    Last_Purchase_Date=('InvoiceDate', 'max'),
    UniqueProducts=('StockCode', 'nunique')
).reset_index()

net_model = hist.groupby('Customer ID').agg(
    Monetary=('Revenue', 'sum'),
    TotalItems=('Quantity', 'sum')
).reset_index()

# Country lấy từ hóa đơn mua hợp lệ đầu tiên trong khoảng lịch sử
country = (purchases_hist.sort_values('InvoiceDate')
           .groupby('Customer ID', as_index=False)['Country'].first())

customer_model = (base_model
                  .merge(net_model, on='Customer ID', how='left')
                  .merge(country, on='Customer ID', how='left'))

customer_model.to_csv(output_model, index=False)
print('Ngày dự báo t:', prediction_date.date())
print('Khoảng lịch sử:', history_start.date(), '->', (prediction_date-pd.Timedelta(days=1)).date())
print('Điều kiện hoạt động: có mua từ', active_start.date(), 'đến trước t')
print('Số khách đủ điều kiện:', len(customer_model))
print('Đã lưu:', output_model)
print(customer_model.head())

Ngày dự báo t: 2011-06-01
Khoảng lịch sử: 2010-06-01 -> 2011-05-31
Điều kiện hoạt động: có mua từ 2010-12-03 đến trước t
Số khách đủ điều kiện: 2658
Đã lưu: ../data/interim/customer_base_model.csv
   Customer ID  Frequency  ... TotalItems         Country
0      12346.0          2  ...         18  United Kingdom
1      12347.0          4  ...       1626         Iceland
2      12348.0          4  ...       2488         Finland
3      12350.0          1  ...        196          Norway
4      12352.0          6  ...        374          Norway

[5 rows x 8 columns]
